# Phase 3: Workforce Intelligence — Step 3.4: Organization-Wide Skill Gap Rollup

This notebook aggregates the employee-level skill gaps (`data/processed/employee_skill_gaps.csv`) to compute the organization-wide demand for each missing skill.

We calculate the frequency of each missing skill and classify its severity using the following rule:
- **HIGH**: Missing in 100 or more employees.
- **MEDIUM**: Missing in 50 to 99 employees.
- **LOW**: Missing in fewer than 50 employees.

We save the aggregated table to `data/processed/org_skill_gaps.csv`.

In [1]:
import os
import pandas as pd
import numpy as np

proc_dir = os.path.join("data", "processed")
print(f"Processed directory: {os.path.abspath(proc_dir)}")

Processed directory: C:\Users\Harshit Mishra\OneDrive\Desktop\enterprise_hr_ai\data\processed


## 1. Load Employee Skill Gaps

In [ ]:
df_gap = pd.read_csv(os.path.join(proc_dir, "employee_skill_gaps.csv"))
print(f"Employee Skill Gaps Shape: {df_gap.shape}")
print(df_gap.head())

## 2. Aggregate Gaps Org-Wide
We count the number of employees missing each skill and calculate the average importance score.

In [3]:
df_org = df_gap.groupby("missing_skill").agg(
    missing_count=("employee_id", "count"),
    avg_importance_score=("importance_score", "mean")
).reset_index()

df_org = df_org.sort_values(by="missing_count", ascending=False)
print(f"Total Unique Missing Skills: {df_org.shape[0]}")
print(df_org.head(10))

Total Unique Missing Skills: 349
           missing_skill  missing_count  avg_importance_score
289              Science            699              2.433448
71     Critical Thinking            684              3.780585
8       Active Listening            683              3.936223
175          Mathematics            679              2.973932
162  Learning Strategies            676              3.105888
211           Monitoring            676              3.555000
191      Microsoft Excel            668              5.000000
301             Speaking            667              3.847376
209       Microsoft Word            664              4.000000
341              Writing            662              3.573036


## 3. Apply Severity Rule
We classify each missing skill's severity based on the number of employees missing it:
- `missing_count >= 100`: HIGH
- `50 <= missing_count < 100`: MEDIUM
- `missing_count < 50`: LOW

In [4]:
def get_severity(count):
    if count >= 100:
        return "HIGH"
    elif count >= 50:
        return "MEDIUM"
    else:
        return "LOW"

df_org["severity"] = df_org["missing_count"].apply(get_severity)

print("Severity distribution across unique missing skills:")
print(df_org["severity"].value_counts())

print("\nTop 15 HIGH severity missing skills:")
print(df_org[df_org["severity"] == "HIGH"].head(15))

Severity distribution across unique missing skills:
severity
HIGH      154
LOW       121
MEDIUM     74
Name: count, dtype: int64

Top 15 HIGH severity missing skills:
                 missing_skill  missing_count  avg_importance_score severity
289                    Science            699              2.433448     HIGH
71           Critical Thinking            684              3.780585     HIGH
8             Active Listening            683              3.936223     HIGH
175                Mathematics            679              2.973932     HIGH
162        Learning Strategies            676              3.105888     HIGH
211                 Monitoring            676              3.555000     HIGH
191            Microsoft Excel            668              5.000000     HIGH
301                   Speaking            667              3.847376     HIGH
209             Microsoft Word            664              4.000000     HIGH
341                    Writing            662              3.57

## 4. Export Org-Wide Gaps

In [5]:
output_path = os.path.join(proc_dir, "org_skill_gaps.csv")
df_org.to_csv(output_path, index=False)
print(f"Successfully saved org-wide skill gaps to {output_path}")

Successfully saved org-wide skill gaps to data\processed\org_skill_gaps.csv
